# Chapter 19: Model Evaluation, Tuning & Explainability
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksekusi alur evaluasi model standar industri:
1. **Dataset Understanding & Baseline Modeling**.
2. **Evaluasi Metrik Komprehensif** (Regresi & Klasifikasi).
3. **Validasi Silang Khusus**: Stratified K-Fold, Group K-Fold, dan Time Series Split.
4. **Hyperparameter Tuning**: GridSearchCV dan RandomizedSearchCV.
5. **Diagnostik Kurva**: Learning Curves & Validation Curves.
6. **Kalibrasi Probabilitas** & Brier Score.
7. **Model Explainability**: Permutation Feature Importance & Partial Dependence.
8. **Analisis Kegagalan (Error Analysis)** & Responsible AI Audit.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance

# 1. Memuat dataset telemetri armada sensor
df = pd.read_csv('data/sensor_fleet_eval_telemetry.csv')
features = [
    'vibration_amplitude_g', 'operating_temp_c', 'pressure_kpa',
    'power_draw_kw', 'acoustic_noise_db', 'operating_hours'
]
print(f'Dimensi dataset telemetri: {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Perangkat yang terdaftar : {df["device_id"].nunique()} unit')
print('Distribusi target status kerusakan:')
print(df['failure_status'].value_counts())

Dimensi dataset telemetri: 500 baris x 11 kolom
Perangkat yang terdaftar : 10 unit
Distribusi target status kerusakan:
failure_status
1    284
0    216


### Pembagian Data Terisolasi & Validasi Silang (Cross-Validation)
Memisahkan 25% data sebagai test set yang tidak boleh disentuh saat pelatihan atau penyetelan hyperparameter.

In [2]:
X = df[features].values
y = df['failure_status'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
print(f'Ukuran Data Latih: {len(X_train)} sampel | Data Uji: {len(X_test)} sampel')
print(f'Hasil Stratified 5-Fold CV (Data Latih Saja):')
print(f'  Skor F1 per fold : {[round(s, 4) for s in cv_scores]}')
print(f'  Rerata Skor F1   : {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)')

Ukuran Data Latih: 375 sampel | Data Uji: 125 sampel
Hasil Stratified 5-Fold CV (Data Latih Saja):
  Skor F1 per fold : [np.float64(0.8889), np.float64(0.8864), np.float64(0.8989), np.float64(0.8608), np.float64(0.8706)]
  Rerata Skor F1   : 88.11% (+/- 1.36%)


### Evaluasi Akhir pada Set Uji Terisolasi (*Final Test Evaluation*)
Mengukur performa generalisasi model terlatih pada data baru yang belum pernah dilihat sebelumnya.

In [3]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print('Hasil Evaluasi Test Set:')
print(f'  F1-Score Uji   : {f1_score(y_test, y_pred):.4f}')
print(f'  ROC-AUC Uji    : {roc_auc_score(y_test, y_prob):.4f}')
print(f'  Brier Score Uji: {brier_score_loss(y_test, y_prob):.4f}')
print('\nLaporan Klasifikasi Data Uji:')
print(classification_report(y_test, y_pred, digits=4))

Hasil Evaluasi Test Set:
  F1-Score Uji   : 0.8219
  ROC-AUC Uji    : 0.8641
  Brier Score Uji: 0.1517

Laporan Klasifikasi Data Uji:
              precision    recall  f1-score   support

           0     0.7800    0.7222    0.7500        54
           1     0.8000    0.8451    0.8219        71

    accuracy                         0.7920       125
   macro avg     0.7900    0.7836    0.7860       125
weighted avg     0.7914    0.7920    0.7908       125



### Model Explainability: Permutation Feature Importance
Mengukur penurunan F1-score aktual pada data uji saat nilai masing-masing fitur diacak.

In [4]:
perm = permutation_importance(pipe, X_test, y_test, scoring='f1', n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    'Fitur': features,
    'Permutation Drop (F1)': perm.importances_mean
}).sort_values(by='Permutation Drop (F1)', ascending=False)
print('Peringkat Permutation Importance pada Data Uji:')
print(perm_df.to_string(index=False))

Peringkat Permutation Importance pada Data Uji:
                Fitur  Permutation Drop (F1)
vibration_amplitude_g               0.024182
        power_draw_kw               0.015253
    acoustic_noise_db               0.011590
         pressure_kpa               0.007346
      operating_hours               0.000708
     operating_temp_c              -0.002644
